# Database Management Systems: Week 6 - In-Depth Notes

## Week 6 Overview: Advanced Normal Forms, Algorithms, and Design Case Study

Week 6 completes the formal theory of relational database design by:

1. **Formalizing Normal Forms** (Module 26): Systematically defining 1NF, 2NF, 3NF, and understanding their hierarchy.
2. **Decomposition Algorithms** (Module 27): Providing polynomial-time algorithms to decompose into 3NF and BCNF, analyzing their properties.
3. **Case Study** (Module 28): Applying the entire design process to a Library Information System from specification to final schema.
4. **Multivalued Dependencies and 4NF** (Module 29): Extending the theory beyond functional dependencies to handle multi-valued attributes.
5. **Design Summary and Temporal Data** (Module 30): Reflecting on the design process, introducing denormalization and temporal database concepts.

This week bridges theory and practice, providing the tools and judgment needed for real-world database design.

---

## Module 26: Normal Forms – 1NF, 2NF, and 3NF

### 26.1. The Normalization Hierarchy

Normalization is a **refinement technique** that organizes data in a database to satisfy key properties. The goal is to eliminate redundancy and anomalies. This is achieved through **decomposition**—splitting large tables into smaller ones.

**Hierarchy of Normal Forms (from transcript):**

```
Unnormalized
    ↓ (decompose for atomicity)
1NF (First Normal Form)
    ↓ (remove partial dependencies)
2NF (Second Normal Form)
    ↓ (remove transitive dependencies)
3NF (Third Normal Form)
    ↓ (stronger restriction)
BCNF (Boyce-Codd Normal Form)
    ↓ (handle multi-valued dependencies)
4NF (Fourth Normal Form)
```

**Key Properties to Maintain During Decomposition:**
1. **Lossless Join**: Must be able to reconstruct original relation via natural join.
2. **Dependency Preservation**: Must be able to check all dependencies locally without joins.

### 26.2. First Normal Form (1NF) – Review

**Definition:** A relation is in 1NF if:
- All underlying domains contain **atomic values**.
- No attribute is **multi-valued**.

**Example of 1NF Violation and Conversion:**

Original (not 1NF):
```
student(SID, Sname, CourseNames)
```
where `CourseNames` is a set of courses (multi-valued).

Converted to 1NF:
```
student(SID, Sname, CourseName)
```
Now each tuple has a single course. The primary key becomes `(SID, CourseName)`.

**Limitation of 1NF:** Even though the relation is now in 1NF, it still suffers from **redundancy** and **anomalies** because of **partial dependencies**.

### 26.3. Second Normal Form (2NF)

**Motivation:** A relation in 1NF may still have redundancy caused by **partial dependencies**.

**Definition of Partial Dependency:**
Let R be a relation schema, X a candidate key of R (could be composite), Y a **proper subset** of X, and A a **non-prime attribute** (not part of any candidate key). If `Y → A` holds, then this is a **partial dependency**.

**Formal Definition of 2NF:**
A relation is in 2NF if:
1. It is in 1NF.
2. It contains **no partial dependencies**.

**Example of Partial Dependency:**

Consider `student_info(SID, Sname, CourseName)` with FDs:
- `SID → Sname`
- `SID, CourseName` is the primary key.

Here:
- `SID` is a proper subset of the key `(SID, CourseName)`.
- `Sname` is a non-prime attribute.
- So `SID → Sname` is a partial dependency.

This causes redundancy: `Sname` is repeated for every course a student takes.

**Decomposition to 2NF:**
Split into:
```
student(SID, Sname)         -- key: SID
enrollment(SID, CourseName) -- key: (SID, CourseName)
```
Now no partial dependencies exist. The join on `SID` is lossless because `SID` is a key in `student`.

**Supplier Example (from transcript):**
Original: `supplier(SID, status, city, PID, quantity)`
FDs: `SID → city`, `city → status`, `SID, PID → quantity`.

Partial dependencies:
- `SID → city` (SID is subset of key, city non-prime)
- `SID → status` (transitive via city, but also partial because SID subset of key)

Decompose to 2NF:
```
supplier_city(SID, city)
part_qty(SID, PID, quantity)
```
This removes partial dependencies. However, `city → status` still causes a transitive dependency (discussed next).

### 26.4. Third Normal Form (3NF)

**Motivation:** Even after 2NF, redundancy may persist due to **transitive dependencies**.

**Definition of Transitive Dependency:**
If `A → B`, `B → A` does **not** hold, and `B → C`, then `A → C` is a transitive dependency. In this case, a non-key attribute (`B`) determines another non-key attribute (`C`).

**Formal Definition of 3NF:**
A relation is in 3NF if:
1. It is in 2NF.
2. It contains **no transitive dependencies**.

Equivalently, for every FD `X → A` (where A is a single non-prime attribute), at least one of:
- `A ⊆ X` (trivial), or
- `X` is a superkey, or
- `A` is part of **some candidate key** (prime attribute).

The third condition is what allows 3NF to be **dependency preserving** while reducing most redundancy.

**Example of Transitive Dependency:**
`book(book_id, author, author_nationality)` with FDs:
- `book_id → author`
- `author → author_nationality` (assuming one author per book for simplicity)
- `book_id → author_nationality` is transitive.

Decompose:
```
book_author(book_id, author)
author_nationality(author, nationality)
```
Now no transitive dependency exists. The join is lossless because `author` is a key in `author_nationality`.

**Supplier Example Continued:**
After 2NF decomposition:
```
supplier_city(SID, city)
city_status(city, status)   -- Need to further decompose because city → status
```
Wait, after 2NF we had `supplier_city(SID, city)` and the FD `city → status`. This is a transitive dependency (`SID → city → status`). So decompose into:
```
supplier_city(SID, city)
city_status(city, status)
```
Both are in 3NF.

### 26.5. Why 3NF is the Most Common Normal Form

From the transcript:
- **3NF is a good balance**: It eliminates most redundancy and anomalies while being achievable without sacrificing dependency preservation.
- **BCNF may not preserve dependencies**: There are cases where BCNF decomposition loses the ability to check some FDs locally.
- **Practical prevalence**: 3NF is often the target normalization level in industry because it handles the vast majority of real-world situations.

---

## Module 27: Decomposition Algorithms – 3NF and BCNF

### 27.1. Decomposition into 3NF

**Goal:** Decompose a relation R into a set of relations such that:
- Each relation is in 3NF.
- The decomposition is **lossless join**.
- The decomposition is **dependency preserving**.

**Algorithm (from transcript):**
1. Compute the **canonical cover** Fc of the set F of FDs.
2. For each FD `X → Y` in Fc:
   - Create a relation `Ri = X ∪ Y`.
3. If none of the Ri contains a candidate key of R:
   - Add a relation containing a candidate key of R.
4. Remove redundant relations (those that are subsets of others).

**Why it works:**
- **3NF**: In each Ri, X is a key (since X → Y is in the minimal cover), so no non-trivial FD with non-superkey left side exists.
- **Dependency preserving**: Each FD in Fc has its own relation, so it can be checked locally.
- **Lossless join**: The inclusion of a candidate key ensures losslessness.

**Example from transcript:**
R = `customer_banker_branch(customer_ID, employee_ID, branch_name, type)`
FDs:
- `customer_ID, employee_ID → branch_name, type`
- `employee_ID → branch_name`
- `customer_ID, branch_name → employee_ID`

Canonical cover:
- Remove `branch_name` from first FD (extraneous because `employee_ID → branch_name`).
- Fc = { `customer_ID, employee_ID → type`, `employee_ID → branch_name`, `customer_ID, branch_name → employee_ID` }

Decomposition:
- For `customer_ID, employee_ID → type`: R1 = `(customer_ID, employee_ID, type)`.
- For `employee_ID → branch_name`: R2 = `(employee_ID, branch_name)`.
- For `customer_ID, branch_name → employee_ID`: R3 = `(customer_ID, branch_name, employee_ID)`.

R3 contains a candidate key (customer_ID, employee_ID), so no extra relation needed. R2 is a subset of R3? Actually R2 = (employee_ID, branch_name) is not a subset of R3 = (customer_ID, branch_name, employee_ID) because R3 contains both customer_ID and branch_name. Actually R2 is a subset of R3 (both attributes are in R3). But R3 has customer_ID too. We can keep both; they're both needed for checking FDs separately.

Final 3NF schema:
```
customer_type(customer_ID, employee_ID, type)
employee_branch(employee_ID, branch_name)
customer_banker(customer_ID, branch_name, employee_ID)
```
All in 3NF, lossless join, dependency preserving.

### 27.2. Testing for 3NF

**Question:** Is a given relation R in 3NF?

**Naive approach:** Check all FDs in F+. This is exponential.

**Better approach:**
- For each FD `α → β` in the original F (not F+), check:
  - If `α` is a superkey (using attribute closure), OK.
  - Else, for each attribute A in β, check if A is prime (member of some candidate key).
- This works because if no violation exists in F, no violation in F+.

**Caveat:** The test for "A is prime" requires enumerating candidate keys, which can be expensive (NP-hard in general). However, for small schemas, it's practical.

### 27.3. Decomposition into BCNF

**Goal:** Decompose R into BCNF relations with lossless join (dependency preservation not guaranteed).

**Algorithm:**
1. If R is in BCNF, stop.
2. Find a non-trivial FD `α → β` in F+ that violates BCNF (α is not a superkey).
3. Decompose R into:
   - R1 = α ∪ β
   - R2 = R - (β - α)
4. Recursively apply to R1 and R2.

**Properties:**
- **Lossless join**: Guaranteed because `α = R1 ∩ R2` is a key of R1.
- **Dependency preservation**: **Not guaranteed**.

### 27.4. Testing for BCNF

**For the original relation R:**
- Check each FD in F (original set, not F+): if left side is not a superkey, violation.
- If no violation in F, then no violation in F+ (a nice property).

**For a decomposed relation Ri:**
- You **cannot** just use the original F; you must consider the **restriction** of F+ to Ri.
- **Algorithm using attribute closure:**
  For each subset α of Ri, compute α+ (under F).
  If α+ includes all attributes of Ri, α is a superkey; OK.
  Else, if α+ includes any attribute from Ri - α, then α determines that attribute; if α is not a superkey, this is a BCNF violation.

### 27.5. Example of BCNF Decomposition Not Preserving Dependencies

From transcript:
R = `(A, B, C)` with F = {A → B, B → C}.
Candidate key: A.
- `A → B` is fine (A is superkey).
- `B → C` violates BCNF (B not superkey).

BCNF decomposition:
- R1 = BC (from B → C)
- R2 = AB

Now, the dependency `A → C` (which is in F+ by transitivity) cannot be checked on either R1 or R2 because it involves attributes in both. So this BCNF decomposition is **not dependency preserving**.

### 27.6. Comparison: 3NF vs BCNF

| Property | 3NF | BCNF |
|---|---|---|
| Redundancy | Some allowed | Zero (with respect to FDs) |
| Dependency preservation | Always possible | May not be possible |
| Lossless join | Always possible | Always possible |
| Key focus | Primary key / candidate key | Candidate key |

**When to choose:**
- Prefer **BCNF** if dependency preservation is not critical.
- Prefer **3NF** if dependency preservation is essential.

---

## Module 28: Case Study – Library Information System (LIS)

This module demonstrates the complete design process from specification to final relational schema.

### 28.1. Specification Summary

**Entities:**
- **Book**: title, author (first name, last name), publisher, year, ISBN (unique for publication), accession number (unique per copy).
- **Member**: membership number (unique), member type (UG, PG, research scholar, faculty), quota.
- **Student**: roll number (unique), membership number, name (first, last), department, degree (UG/PG/PhD), date of birth, mobile (nullable).
- **Faculty**: ID (unique), membership number, name (first, last), department, designation, date of joining, mobile (nullable).
- **Quota**: member type, max books, max duration.

**Relationships:**
- **Issue**: Member issues a Book. Attributes: date of issue.

**Constraints:**
- A book may be issued to a member only if not already issued.
- A book may not be issued to a member if another copy of the book is already issued to the member.
- No issue if the member has exceeded quota.
- No issue if any prior issue by the member has exceeded its duration.

### 28.2. Design Process Steps

1. **Identify Entity Sets and Attributes**: Extract from specification.
2. **Identify Relationships**: Issue relationship between Member and Book.
3. **Transform to Relational Schema**: Create initial tables.
4. **Identify Functional Dependencies**: Hidden in natural language statements.
5. **Normalize to BCNF/3NF**: Decompose as needed.
6. **Refine for Query Efficiency**: Adjust design to support common queries.

### 28.3. Initial Relational Schemas (from ER model)

```
book(accession_no, isbn_no, title, author_fname, author_lname, publisher, year)
member(member_no, member_type)
student(roll_no, member_no, name_fname, name_lname, dept, degree, dob, mobile)
faculty(id, member_no, name_fname, name_lname, dept, designation, doj, mobile)
quota(member_type, max_books, max_duration)
book_issue(member_no, accession_no, date_of_issue)
```

### 28.4. Functional Dependencies (Extracted)

For **book**:
- `isbn_no → (title, author_fname, author_lname, publisher, year)`
- `accession_no → (isbn_no, title, author_fname, author_lname, publisher, year)` (trivially, as key)

For **book_issue**:
- `(member_no, accession_no) → date_of_issue`

For **quota**:
- `member_type → (max_books, max_duration)`

For **member**:
- `member_no → member_type`

For **student**:
- `roll_no → (member_no, name_fname, name_lname, dept, degree, dob, mobile)`
- `member_no → roll_no` (one-to-one)

For **faculty**:
- `id → (member_no, name_fname, name_lname, dept, designation, doj, mobile)`
- `member_no → id` (one-to-one)

### 28.5. Normalization to BCNF

**Book table:** Violates BCNF because `isbn_no` is not a superkey. Decompose:
```
book_copy(accession_no, isbn_no)
book_info(isbn_no, title, author_fname, author_lname, publisher, year)
```
Both in BCNF; lossless join (isbn_no is key in book_info); dependency preserving.

**Other tables:** All in BCNF already (each determinant is a key).

### 28.6. Refinement for Query Efficiency

**Problem:** To find member details for a book issue, you need to know whether the member is a student or faculty. The `member_no` alone doesn't tell you which table to query.

**Solution:** Extend the `member` relation:
```
member(member_no, member_type, member_class, roll_no, id)
```
where:
- `member_class` = 'student' or 'faculty' or 'staff'
- `roll_no` is non-null for students, `id` is non-null for faculty.

This allows efficient querying:
```sql
SELECT ... 
FROM book_issue b, member m, student s
WHERE b.member_no = m.member_no AND m.member_class = 'student' AND s.roll_no = m.roll_no
UNION
SELECT ... 
FROM book_issue b, member m, faculty f
WHERE b.member_no = m.member_no AND m.member_class = 'faculty' AND f.id = m.id
```

**Other refinements:**
- `student` and `faculty` no longer need `member_no` because the `member` table handles membership information.

### 28.7. Final Schema

```
book_copy(accession_no, isbn_no)
book_info(isbn_no, title, author_fname, author_lname, publisher, year)
member(member_no, member_type, member_class, roll_no, id)
quota(member_type, max_books, max_duration)
student(roll_no, name_fname, name_lname, dept, degree, dob, mobile)
faculty(id, name_fname, name_lname, dept, designation, doj, mobile)
book_issue(member_no, accession_no, date_of_issue)
```

All in BCNF, lossless, dependency preserving (with some redundancy possibly, but 3NF is acceptable if needed).

---

## Module 29: Multivalued Dependencies and Fourth Normal Form (4NF)

### 29.1. The Problem: Redundancy Not Caught by FDs

Consider a relation `person(name, phone, dog_like)` where:
- A person can have multiple phones.
- A person can like multiple dog breeds.

This relation has no non-trivial FDs (the key is all attributes). It is in BCNF, but it has massive redundancy: to store two phones and two dog breeds for a person, you must store 2×2 = 4 tuples (Cartesian product).

This redundancy is caused by **multivalued dependencies (MVDs)**, not functional dependencies.

### 29.2. Formal Definition of MVD

**Notation:** `α ↠ β` (read: α multi-determines β).

**Definition:** Given R, α and β subsets of R, `α ↠ β` holds if, for any two tuples t1 and t2 that match on α, there exist tuples t3 and t4 such that:
- t3[α] = t4[α] = t1[α] = t2[α]
- t3[β] = t1[β] and t3[R-β] = t2[R-β]
- t4[β] = t2[β] and t4[R-β] = t1[R-β]

In simpler terms: if two tuples have the same α value, then the β values and the remaining (R-β) values can be independently swapped to form two new valid tuples. This implies that the β values and (R-β) values are **independent** given α.

**Key property:** If `α ↠ β`, then `α ↠ (R - β)` also holds (complement).

### 29.3. Relationship Between FDs and MVDs

- If `α → β` (functional dependency), then `α ↠ β` (multivalued dependency).
- The reverse is not true.

So MVDs are a **generalization** of FDs.

### 29.4. Trivial MVDs

An MVD `α ↠ β` is **trivial** if:
- `β ⊆ α` (like trivial FD), **or**
- `α ∪ β = R` (i.e., β plus α cover all attributes). This is trivial because if α ∪ β = R, then the complement R - β is part of α, so swapping doesn't change anything.

### 29.5. Fourth Normal Form (4NF)

**Definition:** A relation R is in 4NF if for every MVD `α ↠ β` in D+ (the closure of FDs and MVDs), at least one holds:
- `α ↠ β` is trivial, **or**
- `α` is a superkey.

**Consequence:** 4NF implies BCNF. Any relation in 4NF is also in BCNF (because every FD is also an MVD). But BCNF does not imply 4NF.

### 29.6. Decomposition into 4NF

**Algorithm:** Same as BCNF but using MVDs instead of FDs.

Given a relation not in 4NF (violating MVD `A ↠ B` where A is not a superkey):
- Decompose into:
  - R1 = A ∪ B
  - R2 = R - (B - A)

This is lossless join because A is a key of R1.

**Example:**
`person(name, phone, dog_like)` with MVDs `name ↠ phone` and `name ↠ dog_like`.

Since `name` is not a superkey (the key is all three), this violates 4NF.

Decompose using `name ↠ phone`:
- R1 = `(name, phone)`
- R2 = `(name, dog_like)`

Both are in 4NF (each has a trivial MVD with a superkey left side). Redundancy eliminated.

### 29.7. Restrictions for Testing 4NF

To test if a decomposed relation Ri is in 4NF, we must consider the **restriction** of the dependency set D to Ri:
- All FDs in D+ whose attributes are in Ri.
- All MVDs of the form `A ↠ B ∩ Ri` where A ⊆ Ri and the MVD is in D+.

Then check for 4NF within this restricted set.

### 29.8. Extended Example

Given R(A,B,C,G,H,I) with:
- FDs: A → B
- MVDs: C ↠ G

Step-by-step 4NF decomposition as shown in transcript, yielding:
- R1 = (A, B)
- R2 = (C, G, H)
- R3 = (C, G, I)
- R4 = (A, C, G)

This decomposition is lossless, may or may not preserve all dependencies.

### 29.9. Practical Note

4NF is less frequently used than 3NF/BCNF because multi-valued attributes are less common or are handled separately (e.g., separate tables for phone numbers). However, understanding MVDs is important for recognizing when a BCNF relation still has redundancy.

---

## Module 30: Design Summary and Temporal Data

### 30.1. Summary of the Design Process

**Overall Goals:**
- Normalize to BCNF or 4NF with lossless join and dependency preservation.
- If dependency preservation conflicts with BCNF, fall back to 3NF.
- If dependency preservation not critical, BCNF/4NF is preferred.

**Starting Points for Design:**
1. **From ER Model**: Convert entities, relationships, constraints to relational schemas.
2. **From Universal Relation**: Start with all attributes and decompose.
3. **From Ad-hoc Schemas**: Existing tables that need normalization.

**Process:**
1. Identify FDs (from business rules).
2. Compute canonical cover.
3. Decompose to target normal form (3NF or BCNF).
4. Verify lossless join and dependency preservation.

### 30.2. Practical Design Considerations

#### 30.2.1. Denormalization for Performance

Normalization reduces redundancy but can lead to **performance issues** due to joins. Sometimes, **denormalization** (deliberately introducing redundancy) is beneficial for read-heavy applications.

**Example from transcript:** Combining `course` and `prereq` into one table improves lookup speed but increases update cost and storage.

**Trade-offs:**
- Denormalized: Faster queries, slower updates, more storage, potential inconsistency.
- Normalized: Slower queries (due to joins), faster updates, less storage, consistent.

**Alternatives to denormalization:**
- Use **materialized views** to pre-compute joins.
- Use indexes to speed up join queries.

#### 30.2.2. Avoiding Bad Design Patterns

- **Year-wise tables**: Creating separate tables for each year (e.g., `earnings_2004`, `earnings_2005`). This makes cross-year queries very hard.
- **Crosstab**: Representing values as columns (e.g., `earning_2004`, `earning_2005` columns). This is inflexible and causes nulls.

**Better:** Keep a single table with a `year` attribute. Use indexing on year for efficient filtering.

### 30.3. Temporal Databases

**Motivation:** Most real-world data is **time-varying**. Standard relational databases store only the current snapshot; historical information is lost when updates occur.

**Examples:**
- Medical records (blood pressure over time)
- Stock market prices
- Exchange rates
- Address changes

**Basic Concept:**
- Each fact is associated with a **valid time interval** during which it is true.
- A snapshot at time t shows only facts valid at t.

#### 30.3.1. Two Time Dimensions

- **Valid Time**: The time period in the real world during which a fact is true. Example: John lived in Chennai from 1992 to 2015.
- **Transaction Time**: The time period during which the fact is stored in the database. Example: The address change was recorded on January 10, 2016 (though it was valid from June 21, 2015).

**Uni-temporal relations**: Track either valid time or transaction time.
**Bitemporal relations**: Track both.

#### 30.3.2. Example (from transcript)

John's life:
- Born April 3, 1992 in Chennai.
- Birth registered April 6, 1992.
- Moved to Mumbai June 21, 2015.
- Address change registered January 10, 2016.

**Bitemporal representation:**

| Attribute | Valid From | Valid To | Transaction From | Transaction To |
|---|---|---|---|---|
| Chennai | 1992-04-03 | 2015-06-20 | 1992-04-06 | 2016-01-10 |
| Mumbai | 2015-06-21 | ∞ | 2016-01-10 | ∞ |

This allows answering:
- "Where did John live on July 1, 2014?" → Chennai (valid time query).
- "What was the database's record of John's address on June 1, 2015?" → Still Chennai (transaction time query).

#### 30.3.3. Advantages and Disadvantages of Bitemporal Relations

**Advantages:**
- Provides historical data (valid time).
- Provides rollback/audit information (transaction time).

**Disadvantages:**
- More storage.
- More complex queries (need range checks).
- Backup and recovery more complex.

### 30.4. Further Reading

Temporal databases are still an active research area. Extensions of ER model, functional dependencies, and query languages for temporal data are being developed. Standards like SQL:2011 have some temporal support (system-versioned tables), but full temporal modeling is not yet standardized.

---

## Summary of Week 6

In Week 6, we have:

1. **Formalized the hierarchy of normal forms** (1NF → 2NF → 3NF → BCNF → 4NF), understanding each form's conditions and the anomalies they eliminate.

2. **Learned polynomial-time algorithms** for decomposing into 3NF and BCNF, and analyzed their properties (lossless join always guaranteed, dependency preservation for 3NF but not necessarily BCNF).

3. **Applied the design process** to a Library Information System, extracting entities, attributes, relationships, FDs, normalizing, and refining for query efficiency.

4. **Extended the theory to multivalued dependencies and 4NF**, handling redundancy that functional dependencies cannot capture.

5. **Summarized the design process and introduced temporal databases**, providing perspective on practical trade-offs (denormalization) and advanced topics (time-varying data).

This completes the core of relational database design theory. With this foundation, we can now move to other aspects of DBMS, such as storage, indexing, query processing, and transaction management.